# ACT vs ACM 학습 + 평가 - Transfer Cube

> **setup.ipynb를 먼저 실행한 상태여야 합니다.**

### 이 노트북이 하는 일
- ACT (Transformer 기반) 정책 학습
- ACM (Mamba 기반) 정책 학습
- 학습 중 주기적으로 시뮬레이션 평가 (성공률/보상 측정)

### 바꿀 수 있는 설정 (셀 1)
| 변수 | 설명 | 테스트용 | 풀 학습 |
|------|------|----------|---------|
| `STEPS` | 총 학습 스텝 | `2000` | `100000` |
| `SAVE_FREQ` | 체크포인트 저장 주기 | `1000` | `20000` |
| `EVAL_FREQ` | 평가 주기 | `1000` | `10000` |
| `EVAL_EPISODES` | 평가 시 에피소드 수 | `10` | `10` |
| `BATCH_SIZE` | 배치 크기 | `8` | `8` |

### 실행 흐름
셀 0 (경로) → 셀 1 (설정) → 셀 2 (ACT 학습) → 셀 3 (ACM 학습) → 셀 4 (결과 확인)

---
## 0. 경로 설정 + 렌더링 환경

자동으로 setup.ipynb에서 설치한 경로와 OSMesa 렌더링을 설정합니다.
- 수정할 것 없음 (setup.ipynb 설정을 그대로 사용)
- `Rendering: osmesa` 출력 확인

In [1]:
BASE_DIR = "lerobot_project"
VENV_NAME = "lerobot_env"
PYTHON_VERSION = "3.12"
REPO_NAME = "lerobot-ac-mlp"

import os, sys, subprocess
os.environ['MPLBACKEND'] = 'Agg'
home_dir = os.path.expanduser('~')
base_path = os.path.join(home_dir, BASE_DIR)
venv_path = os.path.join(base_path, VENV_NAME)
site_packages = os.path.join(venv_path, 'lib', f'python{PYTHON_VERSION}', 'site-packages')
if site_packages not in sys.path: sys.path.insert(0, site_packages)
python_exec = os.path.join(venv_path, 'bin', 'python')
pip_exec = os.path.join(venv_path, 'bin', 'pip')
project_dir = os.path.join(base_path, REPO_NAME)
src_path = os.path.join(project_dir, 'src')
if src_path not in sys.path: sys.path.insert(0, src_path)

# --- 렌더링 환경 설정 (osmesa) ---
conda_lib = os.path.join(home_dir, 'miniconda3', 'lib')
lib_dirs = {conda_lib}
r = subprocess.run(['bash', '-c', f'find {home_dir}/miniconda3 -name "libOSMesa*" -o -name "libEGL*" 2>/dev/null'],
                    capture_output=True, text=True, timeout=15)
for line in r.stdout.strip().split('\n'):
    if line.strip(): lib_dirs.add(os.path.dirname(line.strip()))
ld_path = ':'.join(lib_dirs) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['LD_LIBRARY_PATH'] = ld_path
os.environ['MUJOCO_GL'] = 'osmesa'

%cd {project_dir}
print(f"Project: {project_dir}")
print(f"Rendering: {os.environ['MUJOCO_GL']}")
!git branch --show-current

/home1/eunji24/lerobot_project/lerobot-ac-mlp
Project: /home1/eunji24/lerobot_project/lerobot-ac-mlp
Rendering: osmesa
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
ecd-eunji


## 1. 실험 설정

학습 파라미터를 설정합니다.
- **테스트**: 기본값 그대로 실행 (STEPS=2000, 몇 분)
- **풀 학습**: 위 표 참고하여 값 변경 (A6000 기준 6~10시간)
- `EVAL_FREQ`: 이 스텝마다 시뮬레이션을 돌려 성공률을 측정합니다

In [2]:
ENV = "aloha"
TASK = "AlohaTransferCube-v0"
DATASET = "lerobot/aloha_sim_transfer_cube_human"

STEPS = 100000          # full: 100000
SAVE_FREQ = 20000      # full: 20000
BATCH_SIZE = 8

# 학습 중 eval 설정 (osmesa 렌더링 사용)
EVAL_FREQ = 20000      # full: 10000
EVAL_EPISODES = 50
EVAL_BATCH = 1

print("실험 설정")
print("=" * 60)
print(f"  환경:     {ENV} / {TASK}")
print(f"  데이터셋: {DATASET}")
print(f"  스텝:     {STEPS},  저장: 매 {SAVE_FREQ}")
print(f"  배치:     {BATCH_SIZE}")
print(f"  평가:     매 {EVAL_FREQ} 스텝, {EVAL_EPISODES} episodes")
print(f"  렌더링:   {os.environ['MUJOCO_GL']}")
print("=" * 60)

실험 설정
  환경:     aloha / AlohaTransferCube-v0
  데이터셋: lerobot/aloha_sim_transfer_cube_human
  스텝:     100000,  저장: 매 20000
  배치:     8
  평가:     매 20000 스텝, 50 episodes
  렌더링:   osmesa


## 2. ACT 학습 (Transformer 디코더)

ACT 정책을 학습합니다. 완료 시 `outputs/train/act_transfer_cube_test/`에 결과 저장.
- 학습 중 로스(loss)가 출력됨 → 숫자가 줄어들면 정상
- eval 시점에 성공률(success rate)도 출력됨
- **이전 결과가 있으면 자동 삭제 후 재학습**

In [5]:
%%time

ACT_OUTPUT = os.path.join(project_dir, "outputs/train/act_transfer_cube_test_100k")

# if os.path.exists(ACT_OUTPUT):
#    import shutil
#    shutil.rmtree(ACT_OUTPUT)
#    print(f"이전 결과 삭제: {ACT_OUTPUT}")

print("ACT 학습 시작 (eval 포함)...")
print("=" * 60)

!MUJOCO_GL={os.environ['MUJOCO_GL']} LD_LIBRARY_PATH={os.environ['LD_LIBRARY_PATH']} \
{python_exec} -m lerobot.scripts.lerobot_train \
    --policy.type=act \
    --env.type={ENV} \
    --env.task={TASK} \
    --dataset.repo_id={DATASET} \
    --dataset.video_backend=pyav \
    --steps={STEPS} \
    --eval_freq={EVAL_FREQ} \
    --save_freq={SAVE_FREQ} \
    --batch_size={BATCH_SIZE} \
    --eval.n_episodes={EVAL_EPISODES} \
    --eval.batch_size={EVAL_BATCH} \
    --output_dir={ACT_OUTPUT} \
    --wandb.enable=true \
    --policy.push_to_hub=false \
    2>&1 | grep -v "Running rollout"

print("=" * 60)
print(f"ACT 학습 완료! 결과: {ACT_OUTPUT}")

이전 결과 삭제: /home1/eunji24/lerobot_project/lerobot-ac-mlp/outputs/train/act_transfer_cube_test_100k
ACT 학습 시작 (eval 포함)...
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
INFO 2026-03-31 22:17:41 ot_train.py:164 {'batch_size': 8,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
   

## 3. ACM 학습 (Mamba 디코더)

ACM 정책을 학습합니다. ACT와 동일한 데이터/설정으로 비교 실험.
- 결과: `outputs/train/acm_transfer_cube_test/`
- ACT보다 빠르거나 비슷한 속도

In [ ]:
%%time

ACM_OUTPUT = os.path.join(project_dir, "outputs/train/acm_ens_test_100k")

"""
if os.path.exists(ACM_OUTPUT):
    import shutil
    shutil.rmtree(ACM_OUTPUT)
    print(f"이전 결과 삭제: {ACM_OUTPUT}")
"""

print("ACM (Mamba) 학습 시작 (eval 포함)...")
print("=" * 60)

!MUJOCO_GL={os.environ['MUJOCO_GL']} LD_LIBRARY_PATH={os.environ['LD_LIBRARY_PATH']} \
{python_exec} -m lerobot.scripts.lerobot_train \
    --policy.type=acm \
    --policy.chunk_size=150 \
    --policy.n_action_steps=80 \
    --env.type={ENV} \
    --env.task={TASK} \
    --dataset.repo_id={DATASET} \
    --dataset.video_backend=pyav \
    --steps={STEPS} \
    --eval_freq={EVAL_FREQ} \
    --save_freq={SAVE_FREQ} \
    --batch_size={BATCH_SIZE} \
    --eval.n_episodes={EVAL_EPISODES} \
    --eval.batch_size={EVAL_BATCH} \
    --output_dir={ACM_OUTPUT} \
    --wandb.enable=true \
    --policy.push_to_hub=false \
    2>&1 | grep -v "Running rollout"

print("=" * 60)
print(f"ACM 학습 완료! 결과: {ACM_OUTPUT}")

ACM (Mamba) 학습 시작 (eval 포함)...
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
INFO 2026-04-08 20:49:21 ot_train.py:164 {'batch_size': 8,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
     

# 추가: eval 돌리기 (ACT)

In [42]:
import os, subprocess, sys

cmd = [
    python_exec,
    "-m", "lerobot.scripts.lerobot_eval",
    "--policy.path=outputs/train/act_transfer_cube_test_100k/checkpoints/060000/pretrained_model",
    "--env.type=aloha",
    "--env.task=AlohaTransferCube-v0",
    "--eval.n_episodes=50",
    "--output_dir=outputs/eval/act_60k"
]

env = os.environ.copy()
env["MUJOCO_GL"] = os.environ["MUJOCO_GL"]
env["LD_LIBRARY_PATH"] = os.environ["LD_LIBRARY_PATH"]

print("CMD:", " ".join(cmd))

proc = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

proc.wait()
print("\nRETURN CODE:", proc.returncode)

CMD: /home1/eunji24/lerobot_project/lerobot_env/bin/python -m lerobot.scripts.lerobot_eval --policy.path=outputs/train/act_transfer_cube_test/checkpoints/002000/pretrained_model --env.type=aloha --env.task=AlohaTransferCube-v0 --eval.n_episodes=50 --output_dir=outputs/eval/acm_test
Traceback (most recent call last):
  File "/home1/eunji24/lerobot_project/lerobot_env/lib/python3.12/site-packages/draccus/parsers/decoding.py", line 159, in decode_dataclass
    instance = origin(**init_args)
               ^^^^^^^^^^^^^^^^^^^
  File "<string>", line 10, in __init__
  File "/home1/eunji24/lerobot_project/lerobot-ac-mlp/src/lerobot/configs/eval.py", line 47, in __post_init__
    self.policy = PreTrainedConfig.from_pretrained(policy_path, cli_overrides=cli_overrides)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home1/eunji24/lerobot_project/lerobot-ac-mlp/src/lerobot/configs/policies.py", line 178, in from_pretrained
    config_file = h

# 추가: eval 돌리기 (ACM)

In [3]:
import os, subprocess, sys

cmd = [
    python_exec,
    "-m", "lerobot.scripts.lerobot_eval",
    "--policy.path=outputs/train/acm_transfer_cube_test_100k/checkpoints/060000/pretrained_model",
    "--policy.temporal_ensemble_coeff=0.01",
    "--env.type=aloha",
    "--env.task=AlohaTransferCube-v0",
    "--eval.n_episodes=50",
    "--output_dir=outputs/eval/acm_test_ensemble",
    
]

env = os.environ.copy()
env["MUJOCO_GL"] = os.environ["MUJOCO_GL"]
env["LD_LIBRARY_PATH"] = os.environ["LD_LIBRARY_PATH"]

print("CMD:", " ".join(cmd))

proc = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

proc.wait()
print("\nRETURN CODE:", proc.returncode)

CMD: /home1/eunji24/lerobot_project/lerobot_env/bin/python -m lerobot.scripts.lerobot_eval --policy.path=outputs/train/acm_transfer_cube_test_100k/checkpoints/060000/pretrained_model --policy.temporal_ensemble_coeff=0.01 --env.type=aloha --env.task=AlohaTransferCube-v0 --eval.n_episodes=50 --output_dir=outputs/eval/acm_test_ensemble
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home1/eunji24/lerobot_project/lerobot-ac-mlp/src/lerobot/scripts/lerobot_eval.py", line 839, in <module>
    main()
  File "/home1/eunji24/lerobot_project/lerobot-ac-mlp/src/lerobot/scripts/lerobot_eval.py", line 835, in main
    eval_main()
  File "/home1/eunji24/lerobot_project/lerobot-ac-mlp/src/lerobot/configs/parser.py", line 232, in wrapper_inner
    cfg = draccus.parse(config_class=argtype, config_path=config_path, args=cli_args)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

## 4. 결과 확인

체크포인트, eval 결과, 영상 목록을 요약합니다.
- 상세 시각화/비교/다운로드 → `results.ipynb`에서 수행

In [16]:
import json
from pathlib import Path

for name, output_dir in [("ACT", ACT_OUTPUT), ("ACM", ACM_OUTPUT)]:
    p = Path(output_dir)
    print(f"\n[{name}]")
    if p.exists():
        # 체크포인트
        ckpts = sorted(p.glob("checkpoints/*/pretrained_model"))
        print(f"  체크포인트: {len(ckpts)}개")
        for c in ckpts:
            print(f"    step {c.parent.name}")

        # eval 결과
        eval_json = p / "eval" / "eval_info.json"
        if not eval_json.exists():
            eval_jsons = sorted(p.rglob("eval_info.json"))
            if eval_jsons:
                eval_json = eval_jsons[-1]

        if eval_json.exists():
            with open(eval_json) as f:
                data = json.load(f)
            info = data.get("overall", data)
            print(f"  평가 결과:")
            for k, v in info.items():
                if k == "video_paths": continue
                print(f"    {k}: {v:.4f}" if isinstance(v, float) else f"    {k}: {v}")

        # 영상
        videos = sorted(p.rglob("*.mp4"))
        if videos:
            print(f"  영상: {len(videos)}개")
            for v in videos[:3]:
                print(f"    {v.name} ({v.stat().st_size/1024/1024:.1f} MB)")
    else:
        print(f"  없음: {output_dir}")

print(f"\n{'='*60}")
print("학습 + 평가 완료!")


[ACT]
  체크포인트: 3개
    step 001000
    step 002000
    step last
  영상: 8개
    eval_episode_0.mp4 (0.2 MB)
    eval_episode_1.mp4 (0.2 MB)
    eval_episode_2.mp4 (0.2 MB)

[ACM]
  체크포인트: 3개
    step 001000
    step 002000
    step last
  영상: 8개
    eval_episode_0.mp4 (0.4 MB)
    eval_episode_1.mp4 (0.3 MB)
    eval_episode_2.mp4 (0.3 MB)

학습 + 평가 완료!


---
## 참고: Full Training 설정

테스트가 정상 동작하면, 실제 실험 값으로 바꿔서 풀 학습을 돌리세요!